In [2]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

In [3]:
import os

print(os.getcwd())

c:\Users\ujjwa\OneDrive\Documents\beauty-recommender\notebooks


In [4]:
os.listdir()

['collaborative.ipynb',
 'content_Recommend.ipynb',
 'data',
 'data_cleaning.ipynb',
 'data_ingestion.ipynb',
 'eda.ipynb',
 'feature_engineering.ipynb',
 'hybrid.ipynb',
 'ML.ipynb',
 'models',
 'popularity_pred.ipynb',
 'project_plan.ipynb',
 'temporary_booster.json']

In [5]:
# ============================================================
# PROJECT PATHS
# ============================================================

project_root = Path.cwd().parent

recommendation_dir = (
    project_root
    / "data"
    / "processed"
    / "recommendation"
)


recommendation_dir = (
    project_root
    / "data"
    / "processed"
    / "recommendation"
)



In [6]:
import pandas as pd

products = pd.read_csv(
    recommendation_dir/"products_processed_popularity.csv",
    low_memory=False
)

reviews = pd.read_csv(
    recommendation_dir/"reviews_processed_popularity.csv",
    low_memory=False
)

In [7]:
print(products.shape)
print(reviews.shape)

(8494, 47)
(1088891, 24)


In [8]:
required_cols = [
    "product_id",
    "product_name",
    "brand_name",
    "primary_category",
    "secondary_category",
    "tertiary_category",
    "price_usd",
    "rating",
    "popularity_score",
    "text_blob"
]

missing = [col for col in required_cols if col not in products.columns]

print("Missing columns:", missing)

Missing columns: []


In [9]:
products.duplicated().sum()

np.int64(0)

In [10]:
products["product_id"].duplicated().sum()

np.int64(0)

In [11]:
products[required_cols].isnull().sum()

product_id            0
product_name          0
brand_name            0
primary_category      0
secondary_category    0
tertiary_category     0
price_usd             0
rating                0
popularity_score      0
text_blob             0
dtype: int64

In [12]:
products["text_blob"] = (
    products["text_blob"]
    .fillna("")
    .astype(str)
)

In [13]:
products["text_blob"] = (
    products["text_blob"]
    .str.lower()
)

In [14]:
import re

products["text_blob"] = (
    products["text_blob"]
    .apply(
        lambda x: re.sub(
            r"[^a-zA-Z ]",
            " ",
            x
        )
    )
)

In [15]:
products["text_blob"] = (
    products["text_blob"]
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
    .str.strip()
)

In [16]:
products[
    ["product_name", "text_blob"]
].head()

,product_name,text_blob
0,Fragrance Discovery Set,fragrance discovery set unisex genderless scen...
1,La Habana Eau de Parfum,la habana eau de parfum unisex genderless scen...
2,Rainbow Bar Eau de Parfum,rainbow bar eau de parfum unisex genderless sc...
3,Kasbah Eau de Parfum,kasbah eau de parfum unisex genderless scent l...
4,Purple Haze Eau de Parfum,purple haze eau de parfum unisex genderless sc...


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [18]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    ngram_range=(1,2)
)

In [19]:
tfidf_matrix = tfidf.fit_transform(
    products["text_blob"]
)

In [20]:
print(tfidf_matrix.shape)

(8494, 10000)


In [21]:
print(type(tfidf_matrix))

<class 'scipy.sparse._csr.csr_matrix'>


In [22]:
len(tfidf.vocabulary_)

10000

In [23]:
feature_names = tfidf.get_feature_names_out()

print(feature_names[:30])

['abbott' 'abeille' 'abeille royale' 'abies' 'abies sibirica' 'absolu'
 'abyssinica' 'abyssinica seed' 'aca' 'aca fruit' 'acacia'
 'acacia decurrens' 'acacia senegal' 'acai' 'acephala' 'acephala kale'
 'acephala leaf' 'acer' 'acer saccharum' 'acerola' 'acerola fruit'
 'acerosa' 'acerosa algae' 'acerosa extract' 'acetal' 'acetamide'
 'acetamide mea' 'acetate' 'acetate acetyl' 'acetate aloe']


In [24]:
from sklearn.metrics.pairwise import cosine_similarity

In [25]:
cosine_sim = cosine_similarity(tfidf_matrix)

cosine_sim.shape

(8494, 8494)

In [26]:
indices = pd.Series(
    products.index,
    index=products["product_name"]
).drop_duplicates()

indices.head()

product_name
Fragrance Discovery Set      0
La Habana Eau de Parfum      1
Rainbow Bar Eau de Parfum    2
Kasbah Eau de Parfum         3
Purple Haze Eau de Parfum    4
dtype: int64

In [27]:
indices["Lip Sleeping Mask"]

product_name
Lip Sleeping Mask    6570
Lip Sleeping Mask    6775
dtype: int64

In [28]:
def recommend_similar_products(
    product_name,
    top_n=10
):

    # Check product exists
    if product_name not in indices:
        return "Product not found."

    # Index of selected product
    idx = indices[product_name]

    # Similarity scores
    similarity_scores = list(
        enumerate(
            cosine_sim[idx]
        )
    )

    # Sort by similarity
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Remove the product itself
    similarity_scores = similarity_scores[1:top_n+1]

    # Product indices
    product_indices = [
        i[0]
        for i in similarity_scores
    ]

    recommendations = products.iloc[
        product_indices
    ][[
        "product_name",
        "brand_name",
        "primary_category",
        "price_usd",
        "rating",
        "popularity_score"
    ]].copy()

    recommendations["similarity"] = [
        score
        for _, score in similarity_scores
    ]

    return recommendations.reset_index(drop=True)

In [29]:
products["product_name"].sample(5)

422     Mini The Cream with TFC8 Face Moisturizer
5741         Mini Conditioner for Beautiful Color
6758         Mini Cleansing Wipes - Coconut Water
5145                       Volume Travel Hair Set
4732                Ultra HD Matte Setting Powder
Name: product_name, dtype: object

In [30]:
recommend_similar_products(
    "Lip Sleeping Mask Intense Hydration with Vitamin C",
    top_n=10
)

,product_name,brand_name,primary_category,price_usd,rating,popularity_score,similarity
0,Berries n' Choco Kisses Set,LANEIGE,Skincare,26.0,4.5954,8.598533,0.909099
1,BTS | Amorepacific Lip Sleeping Mask Lip & Po...,LANEIGE,Skincare,35.0,4.3636,7.649752,0.858235
2,Lip Treatment Balm,LANEIGE,Skincare,25.0,4.2128,8.788290,0.676170
3,Midnight to Morning Hydration Set,LANEIGE,Skincare,21.0,4.2632,8.031971,0.669539
4,Lip Glowy Balm,LANEIGE,Skincare,18.0,4.3923,9.978005,0.551064
5,Besties Set,LANEIGE,Skincare,35.0,3.6667,6.529729,0.463843
6,Lunar New Year Collection: MatteTrance Lipstick,PAT McGRATH LABS,Makeup,39.0,5.0000,6.070770,0.413190
7,Too Femme Heart Core Lipstick,Too Faced,Makeup,26.0,4.6250,7.915935,0.409696
8,MatteTrance Lipstick - Divine Rose Collection,PAT McGRATH LABS,Makeup,39.0,3.9444,7.196304,0.395313
9,Lip Butter Balm,Summer Fridays,Skincare,24.0,4.2662,9.661037,0.394101


In [31]:
recommend_similar_products(
    products.iloc[100]["product_name"]
)

,product_name,brand_name,primary_category,price_usd,rating,popularity_score,similarity
0,GENIUS Ultimate Anti-Aging Eye Cream,Algenist,Skincare,74.0,3.7759,6.909677,0.486700
1,GENIUS Liquid Collagen Serum,Algenist,Skincare,115.0,4.0259,8.588606,0.372632
2,Mini GENIUS Liquid Collagen,Algenist,Skincare,25.0,3.4412,6.844246,0.368635
3,10 Day Results Kit,Algenist,Skincare,88.0,4.8023,6.840228,0.365564
4,GENIUS Ultimate Anti-Aging Cream,Algenist,Skincare,112.0,4.2525,7.832067,0.341569
5,Hyaluronic Acid Booster,Paula's Choice,Skincare,39.0,4.4130,7.041341,0.338746
6,GENIUS Sleeping Collagen Moisturizer,Algenist,Skincare,98.0,4.5413,8.339977,0.337511
7,barrier+ Triple Lipid + Collagen Brightening E...,Skinfix,Skincare,54.0,4.7984,6.575225,0.328694
8,DelIKate Try Me Kit,Kate Somerville,Skincare,49.0,5.0000,6.187049,0.313549
9,Signature Moisturizer,ROSE Ingleton MD,Skincare,85.0,4.7000,6.880385,0.302835


In [32]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

In [33]:
products["popularity_norm"] = scaler.fit_transform(
    products[["popularity_score"]]
)

products["rating_norm"] = scaler.fit_transform(
    products[["avg_rating"]]
)

products["recommendation_norm"] = scaler.fit_transform(
    products[["recommendation_rate"]]
)

In [34]:
products.columns

Index(['product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count',
       'rating', 'reviews', 'size', 'variation_type', 'variation_value',
       'ingredients', 'price_usd', 'value_price_usd', 'sale_price_usd',
       'limited_edition', 'new', 'online_only', 'out_of_stock',
       'sephora_exclusive', 'highlights', 'primary_category',
       'secondary_category', 'tertiary_category', 'child_count',
       'child_max_price', 'child_min_price', 'log_price', 'popularity_score',
       'text_blob', 'avg_rating', 'total_reviews', 'recommendation_rate',
       'avg_helpfulness', 'avg_interaction', 'avg_review_length', 'log_loves',
       'log_reviews', 'price_bucket', 'wilson_score', 'log_loves_scaled',
       'avg_rating_scaled', 'log_reviews_scaled', 'recommendation_rate_scaled',
       'wilson_score_scaled', 'avg_helpfulness_scaled',
       'avg_interaction_scaled', 'final_popularity_score', 'popularity_norm',
       'rating_norm', 'recommendation_norm'],
      dtype='object')

In [35]:
products[
    [
        "popularity_norm",
        "rating_norm",
        "recommendation_norm"
    ]
].describe()

,popularity_norm,rating_norm,recommendation_norm
count,8494.000000,8494.000000,8494.000000
mean,0.631553,0.233965,0.225681
std,0.129374,0.381376,0.372087
min,0.000000,0.000000,0.000000
25%,0.563210,0.000000,0.000000
50%,0.637918,0.000000,0.000000
75%,0.712424,0.721710,0.632953
max,1.000000,1.000000,1.000000


In [36]:
def recommend_hybrid(
    product_name,
    top_n=10
):

    if product_name not in indices:
        return "Product not found."

    idx = indices[product_name]

    similarity_scores = list(
        enumerate(cosine_sim[idx])
    )

    hybrid_results = []

    for product_idx, similarity in similarity_scores:

        hybrid_score = (
            0.60 * similarity +
            0.20 * products.loc[product_idx, "popularity_norm"] +
            0.10 * products.loc[product_idx, "rating_norm"] +
            0.10 * products.loc[product_idx, "recommendation_norm"]
        )

        hybrid_results.append(
            (
                product_idx,
                similarity,
                hybrid_score
            )
        )

    hybrid_results = sorted(
        hybrid_results,
        key=lambda x: x[2],
        reverse=True
    )

    # Remove queried product
    hybrid_results = hybrid_results[1:top_n+1]

    recommendation_indices = [
        x[0]
        for x in hybrid_results
    ]

    recommendations = products.loc[
        recommendation_indices,
        [
            "product_name",
            "brand_name",
            "primary_category",
            "price_usd",
            "avg_rating",
            "recommendation_rate",
            "popularity_score"
        ]
    ].copy()

    recommendations["text_similarity"] = [
        x[1]
        for x in hybrid_results
    ]

    recommendations["hybrid_score"] = [
        x[2]
        for x in hybrid_results
    ]

    return recommendations.reset_index(drop=True)

In [37]:
recommend_hybrid(
    "Lip Sleeping Mask Intense Hydration with Vitamin C",
    top_n=10
)

,product_name,brand_name,primary_category,price_usd,avg_rating,recommendation_rate,popularity_score,text_similarity,hybrid_score
0,Berries n' Choco Kisses Set,LANEIGE,Skincare,26.0,4.601093,0.901639,8.598533,0.909099,0.883711
1,BTS | Amorepacific Lip Sleeping Mask Lip & Po...,LANEIGE,Skincare,35.0,4.375000,0.833333,7.649752,0.858235,0.821991
2,Lip Treatment Balm,LANEIGE,Skincare,25.0,4.212796,0.783032,8.788290,0.676170,0.728297
3,Midnight to Morning Hydration Set,LANEIGE,Skincare,21.0,4.179487,0.820513,8.031971,0.669539,0.711577
4,Lip Glowy Balm,LANEIGE,Skincare,18.0,4.390323,0.853765,9.978005,0.551064,0.688747
5,Lip Butter Balm,Summer Fridays,Skincare,24.0,4.317677,0.830914,9.661037,0.394101,0.584200
6,Glow Lip Pop Lip Balm,Glow Recipe,Skincare,22.0,4.510559,0.898137,8.793923,0.387561,0.572715
7,Besties Set,LANEIGE,Skincare,35.0,3.600000,0.600000,6.529729,0.463843,0.523091
8,Sugar Recovery Lip Mask Advanced Therapy,fresh,Skincare,28.0,4.693333,0.945000,8.032580,0.304064,0.515031
9,Sugar Mint Rush Freshening Lip Treatment,fresh,Skincare,25.0,4.083333,0.791667,7.954409,0.283959,0.473799


In [38]:
import pickle
import os

os.makedirs(
    "data/processed/recommendation/content_based",
    exist_ok=True
)


with open(
    "data/processed/recommendation/content_based/tfidf_vectorizer.pkl",
    "wb"
) as f:
    pickle.dump(
        tfidf,
        f
    )

In [39]:
import os

os.makedirs(
    "models/recommendation/content_based",
    exist_ok=True
)

In [40]:
import os

os.makedirs(
    "models/recommendation/content_based",
    exist_ok=True
)

In [41]:
import pickle

with open(
    "models/recommendation/content_based/tfidf_vectorizer.pkl",
    "wb"
) as f:
    pickle.dump(
        tfidf,
        f
    )

In [42]:
with open(
    "models/recommendation/content_based/tfidf_matrix.pkl",
    "wb"
) as f:
    pickle.dump(
        tfidf_matrix,
        f
    )

In [43]:
with open(
    "models/recommendation/content_based/cosine_similarity.pkl",
    "wb"
) as f:
    pickle.dump(
        cosine_sim,
        f
    )

In [44]:
with open(
    "models/recommendation/content_based/product_indices.pkl",
    "wb"
) as f:
    pickle.dump(
        indices,
        f
    )